# GemmaFit: Gemma 4 Function Calling Fine-Tuning (Unsloth)

Fine-tune Gemma 4 E4B for **8 universal biomechanics safety function calls**.

## Configuration
- **Base model**: `google/gemma-4-4b-it` (Unsloth 4-bit quant)
- **LoRA rank**: 32 (targeting FC accuracy, not creative generation)
- **Training data**: 600 synthetic FC samples (`fc_training_data.json`)
- **Target**: Pixel 8 Pro (Tensor G3, 12GB RAM) via llama.cpp GGUF

## Competition Tracks
- Main Track, Health & Sciences, Safety & Trust
- llama.cpp Special Track (deployment target)
- Unsloth Special Track (fine-tuning tool)

In [ ]:
#@title 1. Install Unsloth & Dependencies
%%capture
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

!pip install --no-deps trl peft accelerate bitsandbytes
!pip install --no-deps unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers triton

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title 2. Load Model & Tokenizer (Unsloth 4-bit)
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/gemma-4-4b-it-bnb-4bit"  #@param {type: "string"}
MAX_SEQ_LENGTH = 2048  #@param {type: "integer"}
LORA_RANK = 32  #@param {type: "integer"}

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"LoRA rank: {LORA_RANK}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Define FC Tool Schemas

These 8 tools correspond to the 8 universal safety rules in `implementation_plan.md`.\nThe LLM must learn to select the correct tool and extract the right arguments.

In [ ]:
#@title 3. FC Tool Definitions

FC_TOOLS = [
    {
        "name": "correct_knee_alignment",
        "description": "Alert when knees collapse inward (valgus). Rule #1: D_knee/D_ankle < 0.8.",
        "parameters": {
            "type": "object",
            "properties": {
                "side": {"type": "string", "enum": ["left", "right", "bilateral"]},
                "ratio": {"type": "number", "description": "D_knee/D_ankle ratio"},
                "severity": {"type": "string", "enum": ["mild", "moderate", "severe"]}
            },
            "required": ["side", "ratio", "severity"]
        }
    },
    {
        "name": "correct_spinal_alignment",
        "description": "Alert when spine deviates from neutral. Rules #2 & #8: deviation > 15 deg.",
        "parameters": {
            "type": "object",
            "properties": {
                "deviation": {"type": "number", "description": "Deviation in degrees"},
                "region": {"type": "string", "enum": ["lumbar", "thoracic", "cervical", "full_spine"]}
            },
            "required": ["deviation", "region"]
        }
    },
    {
        "name": "correct_joint_angle",
        "description": "Alert when a joint approaches hyperextension. Rule #3: angle <= 5 deg or >= 175 deg.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint": {"type": "string", "description": "Joint name, e.g. left_knee"},
                "current": {"type": "number", "description": "Current angle in degrees"},
                "safe_range": {"type": "string", "description": "Expected safe range, e.g. 5-175"}
            },
            "required": ["joint", "current", "safe_range"]
        }
    },
    {
        "name": "correct_asymmetry",
        "description": "Alert when bilateral joint angles differ significantly. Rule #4: left-right diff > 10 deg.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint": {"type": "string", "description": "Joint pair, e.g. knee"},
                "left": {"type": "number", "description": "Left side angle"},
                "right": {"type": "number", "description": "Right side angle"}
            },
            "required": ["joint", "left", "right"]
        }
    },
    {
        "name": "warn_com_offset",
        "description": "Alert when center of mass projection falls outside support base. Rule #5.",
        "parameters": {
            "type": "object",
            "properties": {
                "direction": {"type": "string", "enum": ["forward", "backward", "left", "right"]},
                "distance": {"type": "number", "description": "Offset distance as ratio to support polygon radius"}
            },
            "required": ["direction", "distance"]
        }
    },
    {
        "name": "warn_rapid_movement",
        "description": "Alert when joint angular velocity exceeds safe threshold. Rule #6: > 60 deg/s.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint": {"type": "string", "description": "Joint with rapid movement"},
                "velocity": {"type": "number", "description": "Angular velocity in deg/s"}
            },
            "required": ["joint", "velocity"]
            }
    },
    {
        "name": "increase_range_of_motion",
        "description": "Alert when ROM is insufficient. Rule #7: current ROM < 50% of expected safe ROM.",
        "parameters": {
            "type": "object",
            "properties": {
                "joint": {"type": "string", "description": "Joint with limited ROM"},
                "current": {"type": "number", "description": "Current ROM in degrees"},
                "target": {"type": "number", "description": "Expected safe ROM in degrees"}
            },
            "required": ["joint", "current", "target"]
        }
    },
    {
        "name": "positive_reinforcement",
        "description": "Encourage user when no anomalies detected for 30+ consecutive frames. No specific rule.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Current movement pattern label"},
                "primary_muscles": {"type": "array", "items": {"type": "string"},
                               "description": "Estimated primary muscle groups"},
                "streak": {"type": "integer", "description": "Consecutive safe frames"}
            },
            "required": ["pattern", "primary_muscles", "streak"]
        }
    },
]

print(f"Defined {len(FC_TOOLS)} FC tools")
for t in FC_TOOLS:
    print(f"  - {t['name']}")

In [ ]:
#@title 4. Load & Format Training Data
import json
import random
from datasets import Dataset

# Upload fc_training_data.json to Colab before running this cell
DATA_PATH = "/content/fc_training_data.json"  #@param {type: "string"}

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

random.seed(42)
random.shuffle(raw_data)

TRAIN_RATIO = 0.85
split_idx = int(len(raw_data) * TRAIN_RATIO)
train_data = raw_data[:split_idx]
val_data = raw_data[split_idx:]

print(f"Total samples: {len(raw_data)}")
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

# Function name lookup for formatting
FUNC_NAMES = [t["name"] for t in FC_TOOLS]

def format_fc_prompt(sample: dict) -> str:
    """Format input into a Gemma FC prompt."""
    inp = sample["input"]
    anomalies = inp.get("anomalies", [])
    pattern = inp.get("pattern", "unknown")
    phase = inp.get("phase", "unknown")
    primary = inp.get("estimated_primary", [])
    secondary = inp.get("estimated_secondary", [])
    streak = inp.get("streak_frames", 0)
    confidence = inp.get("confidence", 1.0)

    prompt = f"""Analyze this biomechanics data and call the appropriate safety function.

Movement pattern: {pattern}
Phase: {phase}
Estimated primary muscles: {', '.join(primary) if primary else 'none'}
Estimated secondary muscles: {', '.join(secondary) if secondary else 'none'}
Confidence: {confidence:.2f}
"""

    if anomalies:
        prompt += f"\nSafety anomalies detected ({len(anomalies)}):\n"
        for a in anomalies:
            prompt += f"  - Rule {a.get('rule', '?')}: {a.get('description', 'unknown')}"
            for k, v in a.items():
                if k not in ("rule", "description"):
                    prompt += f", {k}={v}"
            prompt += "\n"
    elif streak >= 30:
        prompt += f"\nNo anomalies detected for {streak} consecutive frames.\n"        
    else:
        prompt += "\nNo anomalies detected.\n"

    prompt += "\nAvailable functions:\n"
    for t in FC_TOOLS:
        prompt += f"  {t['name']}: {t['description']}\n"

    prompt += "\nCall the most appropriate function."
    return prompt


def format_fc_response(sample: dict) -> str:
    """Format output as a function call."""
    out = sample["output"]
    func_name = out["function"]
    args = out.get("args", {})

    # Format as tool call JSON
    response = f"```json\n{{\"name\": \"{func_name}\", \"arguments\": {json.dumps(args, ensure_ascii=False)}}}\n```"
    return response


def format_to_chatml(sample: dict) -> dict:
    """Convert to chat-style format for Unsloth/TRL."""
    prompt = format_fc_prompt(sample)
    response = format_fc_response(sample)
    return {
        "conversations": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
    }


train_formatted = [format_to_chatml(s) for s in train_data]
val_formatted = [format_to_chatml(s) for s in val_data]

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"\nFormatted train: {len(train_dataset)}, val: {len(val_dataset)}")
print(f"\nSample prompt:\n{train_formatted[0]['conversations'][0]['content'][:200]}...")
print(f"\nSample response:\n{train_formatted[0]['conversations'][1]['content'][:200]}...")

In [ ]:
#@title 5. Configure SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

EPOCHS = 3  #@param {type: "integer"}
BATCH_SIZE = 4  #@param {type: "integer"}
GRADIENT_ACCUMULATION = 4  #@param {type: "integer"}
LEARNING_RATE = 2e-4  #@param {type: "number"}
WARMUP_STEPS = 10  #@param {type: "integer"}
WEIGHT_DECAY = 0.01  #@param {type: "number"}
MAX_SEQ_LEN = 2048  #@param {type: "integer"}

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="outputs",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    packing=False,
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

print(f"Training config:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} acc = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"  LR: {LEARNING_RATE}")
print(f"  LoRA rank: {LORA_RANK}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")

In [ ]:
#@title 6. Train!
import time

start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Train loss: {trainer_stats.training_loss:.4f}")
print(f"Train samples/sec: {len(train_dataset) * EPOCHS / elapsed:.1f}")

In [ ]:
#@title 7. FC Accuracy Benchmark (Before vs After)
import json
import time

def benchmark_fc_accuracy(model, tokenizer, val_samples, max_samples=50):
    """Test FC accuracy on validation set."""
    FastLanguageModel.for_inference(model)
    correct = 0
    total = min(len(val_samples), max_samples)
    errors = []

    for i, sample in enumerate(val_samples[:total]):
        prompt = format_fc_prompt(sample)
        expected = sample["output"]
        expected_func = expected["function"]

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=False,
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        # Parse response to extract function name
        predicted_func = None
        try:
            # Try JSON format
            if '"name"' in response:
                json_str = response.split("```json")[-1].split("```")[0] if "```" in response else response
                parsed = json.loads(json_str.strip())
                predicted_func = parsed.get("name", None)
        except:
            # Fallback: check if expected function name appears in response
            for fn in FUNC_NAMES:
                if fn in response:
                    predicted_func = fn
                    break

        if predicted_func == expected_func:
            correct += 1
        else:
            errors.append({
                "sample": i,
                "expected": expected_func,
                "predicted": predicted_func,
                "response": response[:200]
            })

    accuracy = correct / total if total > 0 else 0
    return accuracy, errors

accuracy, errors = benchmark_fc_accuracy(model, tokenizer, val_data)
print(f"\nFC Accuracy: {accuracy:.1%} ({int(accuracy * min(len(val_data), 50))}/{min(len(val_data), 50)})")

if errors:
    print(f"\nSample errors (first 5):")
    for e in errors[:5]:
        print(f"  Sample {e['sample']}: expected={e['expected']}, predicted={e['predicted']}")

In [ ]:
#@title 8. Save LoRA Adapter
LORA_OUTPUT = "gemmafit-gemma4-fc-lora"  #@param {type: "string"}

model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)
print(f"LoRA adapter saved to {LORA_OUTPUT}/")

# Also push to Hub if authenticated
HF_REPO = ""  #@param {type: "string"}
if HF_REPO:
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")

In [ ]:
#@title 9. Export to GGUF (for llama.cpp deployment)
# First merge LoRA into base model
MERGED_MODEL = "gemmafit-gemma4-fc-merged"  #@param {type: "string"}

# Merge and save full model
model.save_pretrained_merged(
    MERGED_MODEL,
    tokenizer,
    save_method="merged_16bit",  # Full precision for GGUF conversion
)
print(f"Merged model saved to {MERGED_MODEL}/")

In [ ]:
#@title 10. Quantize to GGUF Q4_K_M (for Pixel 8 Pro)
# Convert merged model to GGUF format using llama.cpp
# This step should be run after downloading the merged model

GGUF_OUTPUT = "gemma4-e4b-q4.gguf"  #@param {type: "string"}

# Method 1: Using llama.cpp conversion script
print("""To convert the merged model to GGUF:

Method 1 — llama.cpp convert:
  git clone https://github.com/ggerganov/llama.cpp
  cd llama.cpp
  python convert_hf_to_gguf.py ../gemmafit-gemma4-fc-merged \\
    --outfile ../models/gemma4-e4b-q4.gguf \\
    --outtype q4_k_m

Method 2 — Using Unsloth's GGUF export:
  model.save_pretrained_gguf(
      "gemmafit-gemma4-fc-gguf",
      tokenizer,
      quantization_method="q4_k_m",
  )

Then deploy to Pixel 8 Pro:
  adb push models/gemma4-e4b-q4.gguf /sdcard/Android/data/com.gemmafit/files/
""")

## Benchmark Results

Record before/after FC accuracy here:

| Metric | Before (Base) | After (Fine-tuned) |
|--------|---------------|-------------------|
| FC Accuracy | _TBD_ | _TBD_ |
| Correct Function | _TBD_ | _TBD_ |
| Correct Args | _TBD_ | _TBD_ |
| Inference Latency | _TBD_ | _TBD_ |